# Manifiesto de vigencia de los códigos electrónicos del BOE

Cuaderno de lectura de la medición `vigencia-boe/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Respalda el artículo [vigencia-codigos-normativos-boe](https://manpla.net/temas/vigencia-codigos-normativos-boe/). Carga el fichero de al lado —o lo descarga del repositorio si se ejecuta fuera de él—, muestra la ficha de procedencia y dibuja una figura con matplotlib a secas. Solo lee; no vuelve a tomar la instantánea: para eso está `generar.py`.

*Reading notebook for this measurement: loads the file next to it, prints the provenance record and draws one figure. Column names are in Spanish; `GLOSARIO.md` gives the English form.*

In [ ]:
import io, json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/vigencia-boe/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")


## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## El dato

In [ ]:
m = leer("manifiesto.csv")
res = json.loads(texto("resumen-fichas.json")); res.pop("marca_tiempo", None)
print(len(m), "códigos ·", len(res), "fichas resumidas")
m[["codigo", "materia", "bloque", "fecha_actualizacion"]].head(15)

## Una figura

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4.5))
m.materia.value_counts().plot.barh(ax=a); a.set_title("códigos por materia"); a.invert_yaxis()
est = pd.Series([v.get("estado_boe", "") for v in res.values()]).value_counts()
est.plot.bar(ax=b, rot=0); b.set_title("estado declarado por el BOE")
plt.tight_layout()